# eCommerce Dataset — Data Preprocessing
**Workflow:** dataset audit, domain validation, duplicates, missing-value treatment, scaling, engineering, dimensionality / feature selection decision, and class imbalance.  
**Target used:** `Churned` (binary classification).  
This notebook uses the supplied project input CSV. Run it from the **project root**, or use the root-finding cell below. Final four-model comparisons belong to Parts D–G.

In [ ]:
from pathlib import Path
import os, json, sys
root = Path.cwd()
if not (root/'src/preprocessing.py').exists() and (root.parent/'src/preprocessing.py').exists():
    root = root.parent
os.chdir(root)
print('Project root:', root)
assert Path('data/raw/eCommerce-dataset.csv').exists(), 'Missing source CSV' 

## Missing values and source-data audit

In [ ]:
import pandas as pd, numpy as np
raw = pd.read_csv('data/raw/eCommerce-dataset.csv')
print('Rows, columns:',raw.shape)
print('Exact duplicates:',raw.duplicated().sum())
print('Missing cells:',int(raw.isna().sum().sum()))
print('Rows with missing cells:',int(raw.isna().any(axis=1).sum()))
display(raw.head(5))
display(raw.isna().sum().loc[lambda s:s>0].sort_values(ascending=False).to_frame('missing_cells'))

In [ ]:
import matplotlib.pyplot as plt
ax=raw.isna().sum().loc[lambda s:s>0].sort_values().plot.barh(figsize=(9,6),title='Original missing values by field')
ax.set_xlabel('Missing observations');plt.tight_layout();plt.show()

## Domain validation and duplicate removal

In [ ]:
from src.preprocessing import load_clean_base
clean_intermediate,audit_base = load_clean_base(Path('data/raw/eCommerce-dataset.csv'), date_columns=[])
print('Exact duplicates removed:',audit_base['exact_duplicate_rows_removed'])
print('Invalid values flagged:')
display(pd.DataFrame(audit_base['invalid_domain_values_replaced_with_missing']).T)
print('Derived features:',audit_base['engineered_feature_columns'])

## Train-fitted imputation and scaling

In [ ]:
from src.preprocessing import main
# Reproduce the bundle CSVs; parameters estimated only on 40,000 training rows.
audit=main(['--input','data/raw/eCommerce-dataset.csv', '--output-dir','data/processed',
            '--task','classification','--target','Churned', '--export-model-ready'])
print('Predictors after preprocessing:',audit['model_ready_feature_count'])
print('Split:',audit['split'])

In [ ]:
full=pd.read_csv('data/processed/fully_cleaned_dataset.csv')
train=pd.read_csv('data/processed/train_cleaned.csv')
test=pd.read_csv('data/processed/test_cleaned.csv')
print('Full processed shape:',full.shape,'Nulls:',full.isna().sum().sum())
print('Class counts in train:',train.churned.value_counts().to_dict())
print('Class counts in test:',test.churned.value_counts().to_dict())
assert full.shape==(50000,41)
assert full.isna().sum().sum()==0

## Feature engineering and dimensionality decisions

In [ ]:
print('Engineered:',audit['engineered_feature_columns'])
print('All retained predictors are numerical; no categorical encoding is required.')
print('Imputation, missing indicators and scaling are fitted on train.')
print('No additional statistical feature selection or dimensionality reduction is applied by default.')
print('Optional SelectKBest and TruncatedSVD are supported by build_model_pipeline.')

## Class distribution and preprocessing comparison

In [ ]:
print('Class 0 / class 1:',audit['class_distribution'])
import runpy
runpy.run_module('src.preprocessing_comparison', run_name='__main__')
sensitivity=json.loads(Path('data/processed/preprocessing_cv_sensitivity.json').read_text())
display(pd.DataFrame(sensitivity['results']).T)
print('These are exploratory 3-fold training-only CV estimates, not final model-selection or holdout results.')

## Modelling handoff
Use `train_cleaned.csv` as features+labels in the complete `build_model_pipeline()`; use `test_cleaned.csv` only for the final unbiased holdout evaluation. Do **not** cross-validate on `fully_cleaned_dataset.csv` or `model_ready_train.csv`, because the scaling/imputation in those exports was fitted on the entire training partition. For a business impact demonstration, compare recall, precision, F1 and ROC-AUC while considering the retention campaign cost. Final feature-selection/balancing decisions must be validated by the group.

**Open questions**: Is the data genuinely observational or synthetic? At what cutoff are behavioural summaries available? Is `Churned` the approved label? These cannot be resolved from the CSV alone.